# Google Drive Access Test

Test authentication and file download from Google Drive folder.

**Folder:** https://drive.google.com/drive/u/0/folders/1ChBWCNiicOsL9VCWDXUbLe3dlm76TtUK


# Google Drive Access & Documentation Upload

This notebook downloads onboarding docs from Google Drive and uploads them to Weaviate.

### Prerequisites

1. **Environment Variables** - Copy `example.env` to `.env` and fill in:
   - `GOOGLE_OAUTH_CLIENT_ID` - From GCP Console
   - `GOOGLE_OAUTH_CLIENT_SECRET` - From GCP Console
   - `GOOGLE_OAUTH_PROJECT_ID` - Your GCP project
   - `GOOGLE_DRIVE_FOLDER_ID` - The Drive folder with docs
   - `WEAVIATE_URL`, `WEAVIATE_API_KEY`, `OPENAI_API_KEY_DATA_BOT`

### Creating OAuth Credentials

1. Go to [Google Cloud Console](https://console.cloud.google.com/) → APIs & Services → Credentials
2. Click **+ CREATE CREDENTIALS** → **OAuth client ID**
3. If prompted, configure the OAuth consent screen first
4. Select **Desktop app** as the application type
5. Copy the **Client ID** and **Client Secret** to your `.env` file

### Running the Notebook

1. First run will open a browser for Google authentication
2. Subsequent runs use cached credentials (saved in `credentials.json`, gitignored)



## Setup


In [1]:
# Install required packages
%pip install PyDrive2 pyyaml python-docx --quiet
print("✅ Packages installed")



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Packages installed


In [2]:
import sys
from pathlib import Path

# Use absolute paths
project_root = Path("/workspaces/py/projects/onboarding_mascot")
repo_root = Path("/workspaces/py")

# Add paths
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(project_root))

# Add virtual env to path
resolve_name = "python_default"
site_packages = repo_root / "dist/export/python/virtualenvs" / resolve_name / "3.10.18/lib/python3.10/site-packages"
if site_packages.exists():
    sys.path.insert(0, str(site_packages))

print("✅ Environment configured")


✅ Environment configured


## Step 1: Authenticate to Google Drive

**Important:** This will open a browser window for authentication.
1. Click the link that appears
2. Sign in with your Google account
3. Grant access to Google Drive
4. Copy the authorization code
5. Paste it back here


In [3]:
import os
import tempfile
import json
from pathlib import Path
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
import yaml
from dotenv import load_dotenv

# Load environment variables from root .env
load_dotenv(Path("/workspaces/py/.env"))

print("🔐 Starting authentication...\n")

# Get OAuth credentials from environment variables
client_id = os.getenv("GOOGLE_OAUTH_CLIENT_ID")
client_secret = os.getenv("GOOGLE_OAUTH_CLIENT_SECRET")
project_id = os.getenv("GOOGLE_OAUTH_PROJECT_ID")

if not all([client_id, client_secret, project_id]):
    raise ValueError(
        "Missing Google OAuth credentials in .env file!\n"
        "Required: GOOGLE_OAUTH_CLIENT_ID, GOOGLE_OAUTH_CLIENT_SECRET, GOOGLE_OAUTH_PROJECT_ID\n"
        "See example.env for reference."
    )

# Build OAuth config from env vars (no JSON file needed!)
client_config = {
    "installed": {
        "client_id": client_id,
        "client_secret": client_secret,
        "project_id": project_id,
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "auth_provider_x509_cert_url": "https://www.googleapis.com/oauth2/v1/certs",
        "redirect_uris": ["http://localhost"]
    }
}

# Create temp file for client config
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(client_config, f)
    temp_client_config = f.name

# Credentials cache file (gitignored)
credentials_cache = project_root / "src" / "rag" / "vector_store_updates" / "credentials.json"

# PyDrive2 settings
settings = {
    "client_config_backend": "file",
    "client_config_file": temp_client_config,
    "save_credentials": True,
    "save_credentials_backend": "file",
    "save_credentials_file": str(credentials_cache),
    "get_refresh_token": True,
    "oauth_scope": ["https://www.googleapis.com/auth/drive.readonly"]
}

# Create temp settings file
with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    yaml.dump(settings, f)
    temp_settings = f.name

# Authenticate
gauth = GoogleAuth(temp_settings)
gauth.LocalWebserverAuth()

# Create Drive instance
drive = GoogleDrive(gauth)

# Clean up temp file
os.unlink(temp_client_config)

print("\n✅ Successfully authenticated to Google Drive!")
print(f"   Credentials cached to: {credentials_cache}")
print("   (You won't need to re-authenticate on subsequent runs)")



/workspaces/py/dist/export/python/virtualenvs/python_default/3.10.18/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


🔐 Starting authentication...



ValueError: Missing Google OAuth credentials in .env file!
Required: GOOGLE_OAUTH_CLIENT_ID, GOOGLE_OAUTH_CLIENT_SECRET, GOOGLE_OAUTH_PROJECT_ID
See example.env for reference.

## Step 2: List Files in the Folder

Check what files are available in the Google Drive folder.


In [22]:
# Folder ID from the URL
FOLDER_ID = "1ChBWCNiicOsL9VCWDXUbLe3dlm76TtUK"

print(f"📂 Listing files in folder: {FOLDER_ID}\n")
print("="*80)

try:
    # List all files in the folder
    file_list = drive.ListFile({
        'q': f"'{FOLDER_ID}' in parents and trashed=false"
    }).GetList()
    
    print(f"\n✅ Found {len(file_list)} files:\n")
    
    for i, file in enumerate(file_list, 1):
        file_type = file['mimeType'].split('.')[-1] if '.' in file['mimeType'] else file['mimeType']
        print(f"{i}. {file['title']}")
        print(f"   ID: {file['id']}")
        print(f"   Type: {file_type}")
        print(f"   Size: {file.get('fileSize', 'N/A')} bytes")
        print()
    
except Exception as e:
    print(f"\n❌ Error accessing folder: {e}")
    print("\nPossible issues:")
    print("1. You don't have access to this folder")
    print("2. The folder ID is incorrect")
    print("3. The folder is in a different Google account")


📂 Listing files in folder: 1ChBWCNiicOsL9VCWDXUbLe3dlm76TtUK


✅ Found 41 files:

1. Onboarding - AI Image Generation
   ID: 1dE3t5eeJQNUtfBbmcXDGZOwC4Kpc4J8TNWFnKxXWvBY
   Type: document
   Size: 2768 bytes

2. Onboarding - Reframe
   ID: 1i0kjlbOue2vI1El0dQnIoDBBtVb3e4LvgUc1DDKydK4
   Type: document
   Size: 3545 bytes

3. Onboarding - Real-Time Collaboration
   ID: 1WESrPYI5uHUe4buXTGjlW12cBahszzKLv9waIOPp5T0
   Type: document
   Size: 3434 bytes

4. Onboarding - Object Shadows
   ID: 1353EfiX2qGU2qYyTtKYfEmzi3wlRB0g5MpleGJnHTm4
   Type: document
   Size: 2302 bytes

5. Onboarding - Adding Textures
   ID: 1fT67KBu76KypJzdlnEoTb1BtYd6Ke2anWQFNNJvO5Qw
   Type: document
   Size: 2412 bytes

6. Onboarding - Alignment
   ID: 1K6pr21e12_8AfAFfIkSUSy0bHIMH5MNayEbvpUvAjUY
   Type: document
   Size: 2340 bytes

7. Onboarding - Change Element Colors
   ID: 1snwYCIqx-vErQRtsV4_X5UKWDyTfEyEkelqO_6UGKW0
   Type: document
   Size: 2867 bytes

8. Onboarding - Image Cropping
   ID: 1KiZywPqS9yA-lXJ

## Step 3: Download All Google Docs

Download all Google Docs from the folder and export them as `.docx` files.


In [23]:
import docx

# Create local directory for downloads
download_dir = Path("/workspaces/py/projects/onboarding_mascot/vector_store_files/Onboarding Docs")
download_dir.mkdir(parents=True, exist_ok=True)

print("📦 Downloading all Google Docs...\n")
print("="*80)

# Filter for Google Docs only
google_docs = [f for f in file_list if f['mimeType'] == 'application/vnd.google-apps.document']

print(f"\n✅ Found {len(google_docs)} Google Docs to download\n")

downloaded_files = []

for i, doc in enumerate(google_docs, 1):
    print(f"\n[{i}/{len(google_docs)}] Downloading: {doc['title']}")
    
    # Export as .docx
    local_path = download_dir / f"{doc['title']}.docx"
    
    try:
        doc.GetContentFile(
            str(local_path),
            mimetype='application/vnd.openxmlformats-officedocument.wordprocessingml.document'
        )
        
        # Verify we can read it
        test_doc = docx.Document(str(local_path))
        text = '\n'.join([p.text for p in test_doc.paragraphs if p.text.strip()])
        
        print(f"   ✅ Downloaded ({len(test_doc.paragraphs)} paragraphs, {len(text)} chars)")
        downloaded_files.append({
            'title': doc['title'],
            'path': local_path,
            'paragraphs': len(test_doc.paragraphs),
            'chars': len(text)
        })
        
    except Exception as e:
        print(f"   ❌ Error: {e}")

print(f"\n{'='*80}")
print(f"\n🎉 Successfully downloaded {len(downloaded_files)}/{len(google_docs)} files!")
print(f"\n📁 Files saved to: {download_dir}")
print("\n📋 Summary:")
for file in downloaded_files:
    print(f"   ✓ {file['title']}.docx")


📦 Downloading all Google Docs...


✅ Found 41 Google Docs to download


[1/41] Downloading: Onboarding - AI Image Generation
   ✅ Downloaded (14 paragraphs, 1087 chars)

[2/41] Downloading: Onboarding - Reframe
   ✅ Downloaded (30 paragraphs, 2332 chars)

[3/41] Downloading: Onboarding - Real-Time Collaboration
   ✅ Downloaded (32 paragraphs, 1611 chars)

[4/41] Downloading: Onboarding - Object Shadows
   ✅ Downloaded (13 paragraphs, 873 chars)

[5/41] Downloading: Onboarding - Adding Textures
   ✅ Downloaded (15 paragraphs, 967 chars)

[6/41] Downloading: Onboarding - Alignment
   ✅ Downloaded (13 paragraphs, 929 chars)

[7/41] Downloading: Onboarding - Change Element Colors
   ✅ Downloaded (22 paragraphs, 1459 chars)

[8/41] Downloading: Onboarding - Image Cropping
   ✅ Downloaded (17 paragraphs, 825 chars)

[9/41] Downloading: Onboarding - Texture Clipping
   ✅ Downloaded (10 paragraphs, 805 chars)

[10/41] Downloading: Onboarding - Adding Backgrounds
   ✅ Downloaded (14 paragraphs,

## Step 4: Chunk Documents

Apply section-based chunking strategy to all downloaded documents.


In [24]:
from typing import List, Dict

def extract_sections_by_font_size(doc, tool_name: str) -> List[Dict[str, str]]:
    """
    Extract logical sections from structured documentation by detecting headings via font size.
    Paragraphs with font size >= 17 are treated as section headings.
    
    Args:
        doc: python-docx Document object
        tool_name: Name of the tool/document
    
    Returns:
        List of chunk dictionaries with tool, section, content, and metadata
    """
    chunks = []
    current_section = "Introduction"
    current_content = []
    
    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        
        # Check if this paragraph is a heading (font size >= 17)
        is_heading = False
        for run in para.runs:
            if run.font.size and run.font.size.pt >= 17:
                is_heading = True
                break
        
        if is_heading:
            # Save previous section if it has content
            if current_content:
                content_text = '\n'.join(current_content).strip()
                # Skip very short introduction sections (just the title)
                if not (current_section == "Introduction" and len(content_text) < 50):
                    chunks.append({
                        'tool': tool_name,
                        'section': current_section,
                        'content': content_text,
                        'metadata': {
                            'type': 'section',
                            'tool': tool_name,
                            'section': current_section
                        }
                    })
            
            # Start new section
            current_section = text
            current_content = []
        else:
            # Add to current section
            current_content.append(text)
    
    # Add final section
    if current_content:
        content_text = '\n'.join(current_content).strip()
        if not (current_section == "Introduction" and len(content_text) < 50):
            chunks.append({
                'tool': tool_name,
                'section': current_section,
                'content': content_text,
                'metadata': {
                    'type': 'section',
                    'tool': tool_name,
                    'section': current_section
                }
            })
    
    return chunks

print("✅ Font-size based chunking function ready")


✅ Font-size based chunking function ready


In [25]:
# Process all downloaded documents and chunk them
print("📦 Processing all downloaded documents...\n")
print("="*80)

all_chunks = []
doc_chunk_summary = []

# Get all .docx files from the download directory
docx_files = sorted(download_dir.glob("*.docx"))

print(f"\n✅ Found {len(docx_files)} documents to process\n")

for i, doc_path in enumerate(docx_files, 1):
    # Extract tool name from filename (remove "Onboarding - " prefix and ".docx" suffix)
    tool_name = doc_path.stem.replace("Onboarding - ", "")
    
    print(f"\n{'='*80}")
    print(f"📄 DOCUMENT {i}/{len(docx_files)}: {tool_name}")
    print(f"{'='*80}")
    
    try:
        # Load document
        doc = docx.Document(str(doc_path))
        
        # Chunk it using font-size detection
        chunks = extract_sections_by_font_size(doc, tool_name)
        
        # Store for later
        all_chunks.extend(chunks)
        doc_chunk_summary.append({
            'tool': tool_name,
            'path': doc_path,
            'chunks': len(chunks)
        })
        
        # Print chunks
        print(f"\n✅ Created {len(chunks)} chunks:\n")
        
        for j, chunk in enumerate(chunks, 1):
            print(f"\n--- Chunk {j}: {chunk['section']} ---")
            print(f"Length: {len(chunk['content'])} chars")
            print(f"\nContent:")
            print(chunk['content'])  # Show full content
            print()
            
    except Exception as e:
        print(f"\n❌ Error processing document: {e}")

print(f"\n{'='*80}")
print(f"{'='*80}")
print(f"\n🎉 SUMMARY")
print(f"{'='*80}")
print(f"\nTotal documents processed: {len(doc_chunk_summary)}")
print(f"Total chunks created: {len(all_chunks)}")
print(f"\nPer-document breakdown:")
for summary in doc_chunk_summary:
    print(f"  • {summary['tool']}: {summary['chunks']} chunks")
    
print(f"\n{'='*80}")


📦 Processing all downloaded documents...


✅ Found 41 documents to process


📄 DOCUMENT 1/41: AI Background Remover

✅ Created 2 chunks:


--- Chunk 1: Tool Description ---
Length: 333 chars

Content:
The AI Background Remover quickly removes the background from any image directly within the Kittl editor. It’s ideal for isolating objects, logos, or subjects while maintaining image quality.
Key Capabilities
Remove backgrounds instantly using AI
Works on any selected image in the editor
Automatically saves the new, clipped version


--- Chunk 2: How to Use the Tool ---
Length: 642 chars

Content:
1. Access the Background Remover
Open your project in the Editor.
In the right-hand Settings panel, go to Image Tools.
Click AI Background Remover.
2. Remove the Background
Select an image on your artboard.
With the image selected, click AI Background Remover in Image Tools.
The AI will automatically remove the background directly on your artboard.
3. Save and Reuse
The new image (with backgroun

## Step 5: Upload to Weaviate Vector Store

Now that we have our chunks, let's upload them to the Weaviate vector database for semantic search.


In [26]:
def create_contextual_chunks(chunks):
    """
    Add contextual metadata to each chunk for better semantic search.
    The context helps the embedding understand what tool/section this is about.
    """
    contextual_chunks = []
    
    for chunk in chunks:
        # Create a contextual version with metadata
        contextual_text = f"""Tool: {chunk['tool']}
Section: {chunk['section']}

Content:
{chunk['content']}"""
        
        contextual_chunks.append({
            'tool': chunk['tool'],
            'section': chunk['section'],
            'original_content': chunk['content'],
            'contextual_content': contextual_text,  # This gets embedded
            'metadata': chunk['metadata']
        })
    
    return contextual_chunks

# Apply contextual chunking to all chunks
all_contextual_chunks = create_contextual_chunks(all_chunks)

print(f"✅ Created {len(all_contextual_chunks)} contextual chunks")
print(f"\n📘 Example contextual chunk:\n")
print(all_contextual_chunks[0]['contextual_content'][:400])


✅ Created 85 contextual chunks

📘 Example contextual chunk:

Tool: AI Background Remover
Section: Tool Description

Content:
The AI Background Remover quickly removes the background from any image directly within the Kittl editor. It’s ideal for isolating objects, logos, or subjects while maintaining image quality.
Key Capabilities
Remove backgrounds instantly using AI
Works on any selected image in the editor
Automatically saves the new, clipped version


In [27]:
import weaviate
import os

# Load environment variables from root .env
from dotenv import load_dotenv
load_dotenv(Path("/workspaces/py/.env"))

# Your Weaviate credentials
WEAVIATE_URL = os.getenv("WEAVIATE_URL")
WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY_DATA_BOT")

# Connect to Weaviate Cloud
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
    headers={
        "X-OpenAI-Api-Key": OPENAI_API_KEY  # For vectorization
    }
)

print("✅ Connected to Weaviate!")
print(f"Cluster: {WEAVIATE_URL}")


UnexpectedStatusCodeError: Meta endpoint! Unexpected status code: 401, with response body: {'code': 401, 'message': 'unauthorized: invalid api key: invalid token'}.

In [ ]:
from weaviate.classes.config import Configure, Property, DataType

# Define your collection schema
collection_name = "MascotHelpArticles"

# Check if collection already exists and delete it (useful for testing)
if weaviate_client.collections.exists(collection_name):
    weaviate_client.collections.delete(collection_name)
    print(f"🗑️  Deleted existing collection: {collection_name}")

# Create the collection
collection = weaviate_client.collections.create(
    name=collection_name,
    description="Onboarding documentation chunks for Kittl tools",
    
    vectorizer_config=Configure.Vectorizer.text2vec_openai(
        model="text-embedding-3-small",
        vectorize_collection_name=False
    ),
    inverted_index_config=Configure.inverted_index(
        bm25_b=0.7,
        bm25_k1=1.25,
        index_null_state=False,
        index_property_length=False,
        index_timestamps=False
    ),
    
    # Properties (metadata for each chunk)
    properties=[
        Property(
            name="tool",
            data_type=DataType.TEXT,
            description="Tool name (e.g., 'Templates Tool', 'Mockups Tool')",
            skip_vectorization=True,  # Don't vectorize metadata
            index_searchable=False
        ),
        Property(
            name="section",
            data_type=DataType.TEXT,
            description="Section name (e.g., 'Tool Description', 'How to Use')",
            skip_vectorization=True,  # Don't vectorize metadata
            index_searchable=False
        ),
        Property(
            name="content",
            data_type=DataType.TEXT,
            description="The actual content of the chunk",
            skip_vectorization=True,  # Don't vectorize, we'll use contextual_content
            index_searchable=False
        ),
        Property(
            name="contextual_content",
            data_type=DataType.TEXT,
            description="Content with metadata for better embedding",
            skip_vectorization=False,
            index_searchable=True
        )
    ]
)

print(f"✅ Collection '{collection_name}' created successfully!")


🗑️  Deleted existing collection: OnboardingDocs
✅ Collection 'OnboardingDocs' created successfully!


/workspaces/py/dist/export/python/virtualenvs/python_default/3.10.18/lib/python3.10/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


In [ ]:
print(f"📦 Uploading {len(all_contextual_chunks)} chunks to Weaviate...\n")

# Get the collection
collection = weaviate_client.collections.get(collection_name)

# Upload chunks in batch
with collection.batch.dynamic() as batch:
    for i, chunk in enumerate(all_contextual_chunks):
        batch.add_object(
            properties={
                "tool": chunk['tool'],
                "section": chunk['section'],
                "content": chunk['original_content'],  # Clean content for retrieval
                "contextual_content": chunk['contextual_content']  # Gets vectorized
            }
        )
        
        # Progress indicator
        if (i + 1) % 10 == 0:
            print(f"  Uploaded {i + 1}/{len(all_contextual_chunks)} chunks...")

print(f"\n✅ Successfully uploaded {len(all_contextual_chunks)} chunks!")

# Verify
total_objects = collection.aggregate.over_all(total_count=True)
print(f"📊 Total objects in collection: {total_objects.total_count}")


📦 Uploading 85 chunks to Weaviate...

  Uploaded 10/85 chunks...
  Uploaded 20/85 chunks...
  Uploaded 30/85 chunks...
  Uploaded 40/85 chunks...
  Uploaded 50/85 chunks...
  Uploaded 60/85 chunks...
  Uploaded 70/85 chunks...
  Uploaded 80/85 chunks...

✅ Successfully uploaded 85 chunks!
📊 Total objects in collection: 85


## Add Manual Q&A Pairs

Use this section to add manually created Q&A pairs to address specific guardrails or common questions that aren't covered in the documentation.


In [ ]:
def test_retrieval(query_text, top_k=5):
    """
    Test semantic search on the uploaded chunks
    """
    collection = weaviate_client.collections.get(collection_name)
    
    # Perform semantic search
    response = collection.query.near_text(
        query=query_text,
        limit=top_k,
        return_metadata=['distance']  # Include similarity scores
    )
    
    print(f"\n🔍 Query: '{query_text}'")
    print(f"📊 Top {top_k} Results:\n")
    print("="*80)
    
    for i, obj in enumerate(response.objects, 1):
        distance = obj.metadata.distance
        similarity = 1 - distance  # Convert distance to similarity
        
        print(f"\nRank {i} | Similarity: {similarity:.4f}")
        print(f"Tool: {obj.properties['tool']}")
        print(f"Section: {obj.properties['section']}")
        print(f"\nContent Preview:\n{obj.properties['content'][:200]}...")
        print(f"\n{'-'*80}")

# Test with a sample query
test_retrieval("How do I remove backgrounds from images?", top_k=3)



🔍 Query: 'How do I remove backgrounds from images?'
📊 Top 3 Results:


Rank 1 | Similarity: 0.5801
Tool: AI Background Remover
Section: How to Use the Tool

Content Preview:
1. Access the Background Remover
Open your project in the Editor.
In the right-hand Settings panel, go to Image Tools.
Click AI Background Remover.
2. Remove the Background
Select an image on your art...

--------------------------------------------------------------------------------

Rank 2 | Similarity: 0.5693
Tool: AI Background Remover
Section: Tool Description

Content Preview:
The AI Background Remover quickly removes the background from any image directly within the Kittl editor. It’s ideal for isolating objects, logos, or subjects while maintaining image quality.
Key Capa...

--------------------------------------------------------------------------------

Rank 3 | Similarity: 0.5133
Tool: Erasing Images
Section: Erasing Images

Content Preview:
The Eraser Tool in Kittl allows you to remove unwanted parts